<a href="https://colab.research.google.com/github/FilipeSchweitzer/ProjetoAplicado2_CDIA_UniSenai/blob/Jahn_branch/IST.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd

df_ident_og = pd.read_excel('IDENTIFICACAO.xlsx')
df_abund_og = pd.read_excel('ABUND.xlsx')

df_ident = df_ident_og[['Compound', 'Compound ID', 'Formula', 'Fragmentation Score', 'Score', 'Isotope Similarity', 'Description']]
df_abund = df_abund_og[['Compound', 'Identifications']]

df = pd.merge(df_ident, df_abund, on='Compound')

csv = df.to_csv('table.csv', index=False)

df = pd.read_csv('table.csv')

COLUMNS = ['Compound', 'Compound ID', 'Formula', 'SMILES', 'Fragmentation Score', 'Score', 'Isotope Similarity', 'Description',
               'Identifications', 'compound_number', 'compound_complexity', 'fingerprint', 'IUPAC_name', 'InChI', 'mass', 'topological']

tabela_final = pd.DataFrame(columns=COLUMNS)

In [ ]:
#slide prof
def Pubchem_prof(name:str):
    import requests

    url = f'https://pubchem.ncbi.nlm.nih.gov/rest/pug/compound/name/{name}/JSON'

    response = requests.get(url, timeout=10)
    code = response.status_code

    if code == 200:
        data = response.json()
        props = data['PC_Compounds'][0]['props']
        return {p['urn']['label']: p['value'] for p in props}

    else:
        match code:
            case 400:
                return f'Error: {code} BAD REQUEST, name might be wrong, please check again!'
            case _:
                return f'Error: {code}'

In [ ]:
def add_to_final(index, info):

    original_table = df.iloc[index].to_list()
    api_search = info

    print(original_table)

    data = []
    for n in range(len(COLUMNS)):
        if n < 3:
            data.append(original_table[n])
        elif n == 3:
            data.append(api_search['SMILES']['sval'])
        elif n > 3 and n <= 9:
            data.append(original_table[n-2])
        else:
            match COLUMNS[n]:
                case 'compound_number':
                    data.append(api_search['Compound']['ival'])
                case 'compound_complexity':
                    data.append(api_search['Compound Complexity']['fval'])
                case 'fingerprint':
                    data.append(api_search['Fingerprint']['binary'])
                case 'IUPAC_name':
                    data.append(api_search['IUPAC Name']['sval'])
                case 'InChI':
                    data.append(api_search['InChI']['sval'])
                case 'mass':
                    data.append(api_search['Mass']['sval'])
                case 'topological':
                    data.append(api_search['Topological']['fval'])

    # adiciona os valores a tabela final
    tabela_final.loc[len(tabela_final)] = data

In [ ]:
from time import sleep
df_size = len(df['Compound'])

found = 0
for n in range(50, 56):
    compound = df.iloc[n]['Description']

    search = Pubchem_prof(compound)

    if search != 'Error: 404':
        found += 1
        print(f'{n+1}.{compound} / found:{found}')
        add_to_final(n, search)
    sleep(0.2)

tabela_final.to_excel('final.csv', index=False)

51.nan / found:1
['4.53_502.1644n', 'CSID129910060', 'C24H27ClN4O6', np.float64(0.0), np.float64(31.9), np.float64(65.2582860225128), nan, np.int64(11)]
52.Adozelesin / found:2
['4.53_502.1644n', 'CSID16736561', 'C30H22N4O4', np.float64(0.0), np.float64(37.0), np.float64(85.6385831456806), 'Adozelesin', np.int64(11)]
55.Adozelesin / found:3
['4.53_502.1644n', 'CSID317091', 'C30H22N4O4', np.float64(0.0), np.float64(37.0), np.float64(85.6385831456806), 'Adozelesin', np.int64(11)]
56.Adozelesin / found:4
['4.53_502.1644n', 'CSID5293205', 'C30H22N4O4', np.float64(0.0), np.float64(37.0), np.float64(85.6385831456806), 'Adozelesin', np.int64(11)]
